# Cross-talk research: does a target gene influence the pathway of a core (disease-causal) gene?

**Inputs: `CORE` (e.g. disease-causal gene) and `TARGET` (gene of interest).** The downstream pathway of each gene is computed **with the other gene removed**, so only routes independent of the other gene count. Output: shared nodes, *interference edges* (TARGET-side node acting on a CORE-pathway node, with residue/mechanism), endpoint agreement (synergistic / antagonistic), and a cross-talk index.

## 1. Parameters

Database locations come from the YAML config (`config.example.yaml` → copy to `target_net.yaml`); parameters here only override it.

In [ ]:
CORE = "FGFR3"        # core / disease-causal gene
TARGET = "HSP90AA1"   # gene of interest (HSP90AA2 is resolved to HSP90AA1 automatically)
DEMO = True
CONFIG = None          # YAML with database paths, e.g. "../target_net.yaml" (None = ./target_net.yaml, ~/.target_net.yaml, or download on demand)
DEPMAP_DIR = None      # overrides databases.depmap.dir in the YAML
TISSUE = None
MAX_DEPTH = MIN_SCORE = HUB_PENALTY = None   # None = values from the YAML (walk: section)
TARGET_NET_DIR = ".."

## 2. Setup

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(TARGET_NET_DIR))
from targetnet.api import research_crosstalk, summarize_crosstalk, show, figure
from targetnet.plots import use_cdn_renderer
use_cdn_renderer()   # plotly.js from CDN -> small .ipynb files
import pandas as pd
pd.set_option("display.max_colwidth", 80); pd.set_option("display.width", 160)
RUN_DIR = os.path.abspath(os.path.join(TARGET_NET_DIR, "runs")); os.makedirs(RUN_DIR, exist_ok=True)

## 3. Run

In [ ]:
x = research_crosstalk(core=CORE, target=TARGET, config=CONFIG, demo=DEMO, depmap=os.path.expanduser(DEPMAP_DIR) if DEPMAP_DIR else None, tissue=TISSUE,
                       max_depth=MAX_DEPTH, min_score=MIN_SCORE, hub_penalty=HUB_PENALTY, workdir=RUN_DIR)
print(summarize_crosstalk(x))

## 4. Interference edges
Each row: a node reachable only from TARGET acting on a node of the CORE pathway (or on CORE itself). `net_effect_of_goi_on_target` = sign(TARGET→from) × edge effect; compared with CORE's own effect on that node → synergistic / antagonistic. `mechanism` distinguishes phosphorylation from chaperone/stability edges; `via_feedback_loop` flags from-nodes that are also upstream of TARGET.

In [ ]:
x["interference"][["from_node","to_node","mechanism","residue","net_effect_of_goi_on_target","causal_effect_on_target","agreement","via_feedback_loop"]]

## 5. Shared nodes and endpoint agreement
Where do the two pathways converge, and do they push the endpoints (TFs / target genes) in the same direction?

In [ ]:
x["shared"].sort_values("hops_from_causal")

In [ ]:
x["endpoints"]

## 6. Views
Three lanes: CORE pathway / shared / TARGET pathway. Thick green = synergistic interference, red = antagonistic, long-dashed = chaperone edge. Click nodes and edges for details.

In [ ]:
figure(x["views"]["crosstalk"]).show()          # Plotly, inline. Full D3 view: show(x["views"]["crosstalk"])

In [ ]:
figure(x["views"]["core_flow"]).show()          # Plotly, inline. Full D3 view: show(x["views"]["core_flow"])

## 7. Interpretation checklist
- A high cross-talk index is only meaningful relative to other core genes: hub-like targets (HSP90, kinases with many substrates) score high against everything. Run the same TARGET against 3–5 unrelated core genes for a baseline.
- Antagonistic edges are the interesting ones for combination therapy (TARGET inhibition may *release* a brake on the CORE pathway).
- Interference edges with a residue are directly testable by phosphoproteomics (CORE inhibitor ± TARGET inhibitor).